In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.precision', 15)  # Increase decimal precision
pd.set_option('display.width', 150)     # Wider display
pd.set_option('display.max_columns', None)  # Show all column

# Phương pháp tiếp tuyến (Newton-Raphson)

Điều kiện:

* $(a,b)$ là khoảng cách ly nghiệm

* $f'(x)$ xác định dấu không đổi trên $[a,b]$

* $f''(x)$ xác định dấu không đổi trên $[a,b]$

Điều kiện hội tụ:

* Chọn đúng điểm xuất phát ban đầu $x_0$ (điểm Fourier - $f(x_0) \cdot f''(x_0) > 0$)

# Thuật toán:

1. Chọn đúng điểm xuất phát ban đầu $x_0$ (điểm Fourier - $f(x_0) \cdot f''(x_0) > 0$)

2. Xác định $x$ theo công thức lặp: $x_{k+1} = x_k - \dfrac{f(x_k)}{f'(x_k)}$

3. Lặp lại bước (2) cho đến khi:

   a. $x_n$ là nghiệm của phương trình, hoặc

   b. dãy $\{x_n\}$ thỏa mãn điều kiện dừng

5. In giá trị $x_n$ (giá trị $x$ sau $n$ lần lặp)

# Áp dụng

## 1. Hậu nghiệm

Kiểm tra sai số theo 1 trong 2 công thức sai số sau:
+ $|x_n - x^*| \leqslant \dfrac{|f(x_n)|}{m_1}$ $\quad\quad$ (1)

+ $|x_n - x^*| \leqslant \dfrac{M_2}{2 m_1} \cdot |x_n - x_{n-1}|^2$ $\quad\quad$ (2)

với $m_1 = \min\limits_{[a,b]} |f'(x)|, \quad M_2 = \max\limits_{[a,b]} |f''(x)|$


### 1.1 Thuật toán
#### Input:
- Khoảng cách ly nghiệm $(a,b)$
- Sai số $\varepsilon$
- Hàm số $f(x)$ thoả mãn $f'(x), f''(x)$ xác định không đổi dấu trên $[a,b]$
- Giá trị nhỏ nhất của $|f'(x)|$ ($m_1$) và giá trị lớn nhất của $|f''(x)|$ ($M_2$) trên $[a,b]$
#### Output: nghiệm gần đúng $x$
#### Thuật toán:
- Bước 1: Lưu dấu của $f''(x)$ vào biến sign. Kiểm tra:
    - Nếu $f(a) \cdot \text{sign} > 0$: chọn $x_0 = a$
    - Nếu $f(a) \cdot \text{sign} < 0$: chọn $x_0 = b$
- Bước 2: Khởi tạo $i=0$. Lặp đến khi thoả mãn điều kiện dừng $|f(x_n)| < m_1 \varepsilon $ hoặc $|x_n - x_{n-1}| < \sqrt{2 \varepsilon \cdot \dfrac{m_1}{M_2}}$:
    - Tính $x_{i+1}=x_i - \dfrac{f(x_i)}{f'(x_i)}, \quad \Delta x_{i}=\dfrac{|f(x_{i+1})|}{m_1}$ hoặc $\Delta x_{i} = \dfrac{M_2}{2 m_1} \cdot |x_{i+1} - x_{i}|^2$

    - Nếu $f(x_{i+1})=0$: thu được nghiệm $x=x_{i+1}$, dừng chương trình
    - Cập nhật $i=i+1$
- Bước 5: Sau khi thoả mãn điều kiện dừng với $n$ lần lặp, thu được nghiệm gần đúng là $x=x_n - \dfrac{f(x_n)}{f'(x_n)}$

### 1.2 Code
#### 1.2.1 Có cộng thêm sai số làm tròn
##### Công thức 1

In [11]:
def newton_iteration_v1 (f, df, d2f, a, b, n, rbl):	
	M2 = max([np.abs(d2f(x)) for x in [a, b]]) #M2 is the maximum value of |f''(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print (f"m1 = {m1}, M2 = {M2}")

	#Starting values
	if f(a) * d2f(a) > 0:
		x = a
	elif f(b) * d2f(b) > 0:
		x = b
	
	results = []
	results.append({
        'n': 0,
        'x_n': x,
        'f(x_n)': f(x),
        'delta_x=|f(x_n)| / m_1': abs(f(x)) / m1
    })

	#Newton's method
	x_new = 0; delta_x = 0
	for i in range(n):
		# calculate the next value of x
		x_new = x - f(x) / df(x)
		delta_x = abs(f(x_new)) / m1

		results.append({
            'n': i+1,
            'x_n': x_new,
            'f(x_n)': f(x_new),
            'delta_x=|f(x_n)| / m_1': delta_x
        })

		# Prepare for next iteration
		x = x_new
		if (f(x_new) == 0): 
			break

	#Print the final result
	df_results = pd.DataFrame(results)
	print(df_results.to_string(index=False))

	if rbl == None:
		print(f"The value of root is: {x}")
	else:
		total_delta = delta_x + 0.5 * 10**(-rbl)
		print(f"The value of root with {rbl} decimal point is: {round(x, rbl)}")
		print(f"Relative error is: {total_delta}")

In [ ]:
f = lambda x: x**4-27
df = lambda x: 4*x**3
d2f = lambda x: 12*x**2

a = 2
b = 3

n = 5
rbl = 9

newton_iteration_v1 (f, df, d2f, a, b, n, rbl)

m1 = 32, M2 = 108
 n               x_n             f(x_n)  delta_x=|f(x_n)| / m_1
 0 3.000000000000000 54.000000000000000       1.687500000000000
 1 2.500000000000000 12.062500000000000       0.376953125000000
 2 2.307000000000000  1.326334418000997       0.041447950562531
 3 2.279994621743364  0.023107580939026       0.000722111904345
 4 2.279507213327209  0.000007408717718       0.000000231522429
 5 2.279507056954794  0.000000000000771       0.000000000000024
The value of root with 9 decimal point is: 2.279507057
Relative error is: 5.000240918396344e-10


#### Công thức 2

In [13]:
def newton_iteration_v2 (f, df, d2f, a, b, n, rbl):	
	M2 = max([np.abs(d2f(x)) for x in [a, b]]) #M2 is the maximum value of |f''(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print (f"m1 = {m1}, M2 = {M2}")

	#Starting values
	if f(a) * d2f(a) > 0:
		x = a
	elif f(b) * d2f(b) > 0:
		x = b
	
	results = []
	results.append({
        'n': 0,
        'x_n': x,
        'f(x_n)': f(x),
        'delta_x=M_2/(2*m_1) * |x_n - x_(n-1)|^2': None
    })

	#Newton's method
	x_new = 0; delta_x = 0;
	for i in range(n):
		# calculate the next value of x
		x_new = x - f(x) / df(x)
		delta_x = M2 / (2 * m1) * (x_new - x)**2

		results.append({
            'n': i+1,
            'x_n': x_new,
            'f(x_n)': f(x_new),
            'delta=M_2/(2*m_1) * |x_n - x_(n-1)|^2': delta_x
        })

		# Prepare for next iteration
		x = x_new
		if (f(x_new) == 0): 
			break

	#Print the final result
	df_results = pd.DataFrame(results)
	print(df_results.to_string(index=False))

	if rbl == None:
		print(f"The value of root is: {x}")
	else:
		total_delta = delta_x + 0.5 * 10**(-rbl)
		print(f"The value of root with {rbl} decimal point is: {round(x, rbl)}")
		print(f"Relative error is: {total_delta}")

In [ ]:
f = lambda x: x**4-27
df = lambda x: 4*x**3
d2f = lambda x: 12*x**2

a = 2
b = 3

n = 5
rbl = 9

newton_iteration_v2 (f, df, d2f, a, b, n, rbl)

m1 = 32, M2 = 108
 n               x_n             f(x_n)  delta_x=M_2/(2*m_1) * |x_n - x_(n-1)|^2  delta=M_2/(2*m_1) * |x_n - x_(n-1)|^2
 0 3.000000000000000 54.000000000000000                                      NaN                                    NaN
 1 2.500000000000000 12.062500000000000                                      NaN                      0.421875000000000
 2 2.307000000000000  1.326334418000997                                      NaN                      0.062857687500000
 3 2.279994621743364  0.023107580939026                                      NaN                      0.001230677642448
 4 2.279507213327209  0.000007408717718                                      NaN                      0.000000400894252
 5 2.279507056954794  0.000000000000771                                      NaN                      0.000000000000041
The value of root with 9 decimal point is: 2.279507057
Relative error is: 5.000412633105506e-10


#### 1.2.2 Không cộng thêm sai số làm tròn

Thiết lập điều kiện dừng theo 1 trong 2 công thức sai số sau:
+ $|x_n - x^*| \leqslant \dfrac{|f(x_n)|}{m_1} < \epsilon$ $\quad\quad (1)$

+ $|x_n - x^*| \leqslant \dfrac{M_2}{2 m_1} \cdot |x_n - x_{n-1}|^2 < \epsilon$ $\quad\quad (2)$

với $m_1 = \min\limits_{[a,b]} |f'(x)|, \quad M_2 = \max\limits_{[a,b]} |f''(x)|$


#### Công thức 1

Từ công thức (1) suy ra Điều kiện dừng: $|f(x_n)| < m_1 \varepsilon$

In [4]:
def newton_recursion_absolute_v1 (f, df, d2f, a, b, eps):	
	M2 = max([np.abs(d2f(x)) for x in [a, b]]) #M2 is the maximum value of |f''(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print (f"m1 = {m1}, M2 = {M2}")

	#Starting values
	if f(a) * d2f(a) > 0:
		x = a
	elif f(b) * d2f(b) > 0:
		x = b
	
	results = []
	#Newton's method
	new_eps = m1*eps
	print(f"delta_x = {new_eps}")

	i=0
	while True:
		# calculate the next value of x
		x_new = x - f(x) / df(x)
		current_delta_x = abs(f(x_new))

		results.append({
            'n': i,
            'x_(n+1)': x_new,
            'f(x_(n+1)': f(x_new),
            'delta_x=|f(x_n)|': current_delta_x
        })
		
		# update the value of interval 
		x = x_new
		if (f(x_new) == 0): 
			break

		#stop condition
		if current_delta_x < new_eps:
			break
		else:
			i += 1

	#Print the final result
	df_results = pd.DataFrame(results)
	print(df_results.to_string(index=False))


	print(f"The value of root with absolute error {eps} is: {x}")

Bài tập VD trên lớp:

In [5]:
f = lambda x: np.exp(x)-4*np.cos(x/2)+1
df = lambda x: np.exp(x)+2*np.sin(x/2)
d2f = lambda x: np.exp(x)+np.cos(x/2)

a = -3
b = -2

eps = 0.5 * pow(10, -7)

newton_recursion_absolute_v1 (f, df, d2f, a, b, eps)

m1 = 1.5476066863791802, M2 = 0.6756375891047525
delta_x = 7.738033431895901e-08
 n            x_(n+1)         f(x_(n+1)  delta_x=|f(x_n)|
 0 -2.605779808477085 0.014993030958223 0.014993030958223
 1 -2.597696479198998 0.000011109692728 0.000011109692728
 2 -2.597690480619852 0.000000000006172 0.000000000006172
The value of root with absolute error 5e-08 is: -2.5976904806198515


Câu 8.1. Viết thuật toán tính gần đúng hằng số $e$ với sai số $\varepsilon$ cho trước. Áp dụng tính $e$ với bảy chữ số đáng tin sau dấu phẩy.

In [17]:
f = lambda x: np.log(x)-1
df = lambda x: 1/x
d2f = lambda x: -1/x**2

a = 2
b = 3
eps = 0.5 * pow(10, -7)
newton_recursion_absolute_v1 (f, df, d2f, a, b, eps)

m1 = 0.3333333333333333, M2 = 0.25
delta_x = 1.6666666666666664e-08
 n           x_(n+1)         f(x_(n+1))  delta_x=|f(x_n)|
 0 2.613705638880109 -0.039231000595620 0.039231000595620
 1 2.716243926355790 -0.000749983454204 0.000749983454204
 2 2.718281064358138 -0.000000281097054 0.000000281097054
 3 2.718281828458938 -0.000000000000040 0.000000000000040
The value of root with absolute error 5e-08 is: 2.7182818284589376


Câu 8.2. Viết thuật toán tính gần đúng hằng số $\pi$ với sai số $\varepsilon$ cho trước. Áp dụng tính $\pi$ với bảy chữ số đáng tin sau dấu phẩy.

In [18]:
f = lambda x: np.tan(x/4)-1
df = lambda x: 1/(4*(np.cos(x/4))**2)
d2f = lambda x: (np.sin(x/4)) / (8*(np.cos(x/4)**3))

a = 3
b = 3.2
eps = 0.5 * pow(10, -7)
newton_recursion_absolute_v1 (f, df, d2f, a, b, eps)

m1 = 0.4669679910450819, M2 = 0.26515194952600585
delta_x = 2.3348399552254093e-08
 n           x_(n+1)        f(x_(n+1))  delta_x=|f(x_n)|
 0 3.142453749314412 0.000430640574651 0.000430640574651
 1 3.141592838987856 0.000000092699036 0.000000092699036
 2 3.141592653589802 0.000000000000004 0.000000000000004
The value of root with absolute error 5e-08 is: 3.141592653589802


Câu 9. Viết thuật toán tính gần đúng $\sqrt[n]{a}, a \in \mathbb{R}, n \in \mathbb{N}$ với sai số $\varepsilon$ cho trước. Áp dụng tính $\sqrt[5]{17}$ với 6 chữ số đáng tin.

In [19]:
f = lambda x: x**5-17
df = lambda x: 5*x**4
d2f = lambda x: 20*x**3

a = 1
b = 2
eps = 0.5 * pow(10, -5)
newton_recursion_absolute_v1 (f, df, d2f, a, b, eps)

m1 = 5, M2 = 160
delta_x = 2.5e-05
 n           x_(n+1)        f(x_(n+1))  delta_x=|f(x_n)|
 0 1.812500000000000 2.560956001281738 2.560956001281738
 1 1.765040839496608 0.130648055177147 0.130648055177147
 2 1.762348598639803 0.000397951056357 0.000397951056357
 3 1.762340347909573 0.000000003726136 0.000000003726136
The value of root with absolute error 5e-06 is: 1.7623403479095725


Câu 10. Tìm nghiệm âm lớn nhất của phương trình $e^x-\cos 2x=0$ với 8 chữ số đáng tin bằng phương pháp tiếp tuyến.

In [20]:
f = lambda x: np.exp(x)-np.cos(2*x)
df = lambda x: np.exp(x)+2*np.sin(2*x)
d2f = lambda x: np.exp(x)+4*np.cos(2*x)


a = -0.6
b = -0.3
eps = 0.5 * pow(10, -8)
newton_recursion_absolute_v1 (f, df, d2f, a, b, eps)

m1 = 0.38846672610835287, M2 = 4.042160680320431
delta_x = 1.9423336305417645e-09
 n            x_(n+1)        f(x_(n+1))  delta_x=|f(x_n)|
 0 -0.458238709389642 0.023777535712125 0.023777535712125
 1 -0.433328474821708 0.000969361674442 0.000969361674442
 2 -0.432221885533905 0.000001983967175 0.000001983967175
 3 -0.432219611394149 0.000000000008392 0.000000000008392
The value of root with absolute error 5e-09 is: -0.4322196113941486


#### Công thức 2

Từ công thức (2) suy ra Điều kiện dừng: $|x_n - x_{n-1}| < \sqrt{2 \varepsilon \cdot \dfrac{m_1}{M_2}}$

In [21]:
def newton_recursion_absolute_v2 (f, df, d2f, a, b, eps):	
	M2 = max([np.abs(d2f(x)) for x in [a, b]]) #M2 is the maximum value of |f''(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print (f"m1 = {m1}, M2 = {M2}")

	#Starting values
	if f(a) * d2f(a) > 0:
		x = a
	elif f(b) * d2f(b) > 0:
		x = b
	
	results = []
	#Newton's method
	new_eps = np.sqrt(eps * 2 * m1 / M2)
	print(f"delta_x = {new_eps}")

	i=0
	while True:
		# calculate the next value of x
		x_new = x - f(x) / df(x)
		current_delta_x = abs(x-x_new)

		results.append({
            'n': i,
            'x_(n+1)': x_new,
            'f(x_(n+1))': f(x_new),
            'delta_x=|f(x_n)|': current_delta_x
        })
		
		# update the value of interval 
		x = x_new
		if (f(x_new) == 0): 
			break

		#stop condition
		if current_delta_x < new_eps:
			break
		else:
			i += 1

	#Print the final result
	df_results = pd.DataFrame(results)
	print(df_results.to_string(index=False))


	print(f"The value of root with absolute error {eps} is: {x}")

Bài tập VD trên lớp:

In [22]:
f = lambda x: np.exp(x)-4*np.cos(x/2)+1
df = lambda x: np.exp(x)+2*np.sin(x/2)
d2f = lambda x: np.exp(x)+np.cos(x/2)

a = -3
b = -2

eps = 0.5 * pow(10, -7)

newton_recursion_absolute_v2 (f, df, d2f, a, b, eps)

m1 = 1.5476066863791802, M2 = 0.6756375891047525
delta_x = 0.0004786007743345305
 n            x_(n+1)        f(x_(n+1))  delta_x=|f(x_n)|
 0 -2.605779808477085 0.014993030958223 0.394220191522915
 1 -2.597696479198998 0.000011109692728 0.008083329278087
 2 -2.597690480619852 0.000000000006172 0.000005998579147
The value of root with absolute error 5e-08 is: -2.5976904806198515


Câu 8.1. Viết thuật toán tính gần đúng hằng số $e$ với sai số $\varepsilon$ cho trước. Áp dụng tính $e$ với bảy chữ số đáng tin sau dấu phẩy.

In [23]:
f = lambda x: np.log(x)-1
df = lambda x: 1/x
d2f = lambda x: -1/x**2

a = 2
b = 3
eps = 0.5 * pow(10, -7)
newton_recursion_absolute_v2 (f, df, d2f, a, b, eps)

m1 = 0.3333333333333333, M2 = 0.25
delta_x = 0.0003651483716701107
 n           x_(n+1)         f(x_(n+1))  delta_x=|f(x_n)|
 0 2.613705638880109 -0.039231000595620 0.613705638880109
 1 2.716243926355790 -0.000749983454204 0.102538287475681
 2 2.718281064358138 -0.000000281097054 0.002037138002348
 3 2.718281828458938 -0.000000000000040 0.000000764100800
The value of root with absolute error 5e-08 is: 2.7182818284589376


Câu 8.2. Viết thuật toán tính gần đúng hằng số $\pi$ với sai số $\varepsilon$ cho trước. Áp dụng tính $\pi$ với bảy chữ số đáng tin sau dấu phẩy.

In [24]:
f = lambda x: np.tan(x/4)-1
df = lambda x: 1/(4*(np.cos(x/4))**2)
d2f = lambda x: (np.sin(x/4)) / (8*(np.cos(x/4)**3))

a = 3
b = 3.2
eps = 0.5 * pow(10, -7)
newton_recursion_absolute_v2 (f, df, d2f, a, b, eps)

m1 = 0.4669679910450819, M2 = 0.26515194952600585
delta_x = 0.00041965861581281567
 n           x_(n+1)        f(x_(n+1))  delta_x=|f(x_n)|
 0 3.142453749314412 0.000430640574651 0.057546250685588
 1 3.141592838987856 0.000000092699036 0.000860910326556
 2 3.141592653589802 0.000000000000004 0.000000185398054
The value of root with absolute error 5e-08 is: 3.141592653589802


Câu 9. Viết thuật toán tính gần đúng $\sqrt[n]{a}, a \in \mathbb{R}, n \in \mathbb{N}$ với sai số $\varepsilon$ cho trước. Áp dụng tính $\sqrt[5]{17}$ với 6 chữ số đáng tin.

In [25]:
f = lambda x: x**5-17
df = lambda x: 5*x**4
d2f = lambda x: 20*x**3

a = 1
b = 2
eps = 0.5 * pow(10, -5)
newton_recursion_absolute_v2 (f, df, d2f, a, b, eps)

m1 = 5, M2 = 160
delta_x = 0.0005590169943749475
 n           x_(n+1)        f(x_(n+1))  delta_x=|f(x_n)|
 0 1.812500000000000 2.560956001281738 0.187500000000000
 1 1.765040839496608 0.130648055177147 0.047459160503392
 2 1.762348598639803 0.000397951056357 0.002692240856805
 3 1.762340347909573 0.000000003726136 0.000008250730230
The value of root with absolute error 5e-06 is: 1.7623403479095725


Câu 10. Tìm nghiệm âm lớn nhất của phương trình $e^x-\cos 2x=0$ với 8 chữ số đáng tin bằng phương pháp tiếp tuyến.

In [26]:
f = lambda x: np.exp(x)-np.cos(2*x)
df = lambda x: np.exp(x)+2*np.sin(2*x)
d2f = lambda x: np.exp(x)+4*np.cos(2*x)


a = -0.6
b = -0.3
eps = 0.5 * pow(10, -8)
newton_recursion_absolute_v2 (f, df, d2f, a, b, eps)

m1 = 0.38846672610835287, M2 = 4.042160680320431
delta_x = 3.100060190513669e-05
 n            x_(n+1)        f(x_(n+1))  delta_x=|f(x_n)|
 0 -0.458238709389642 0.023777535712125 0.141761290610358
 1 -0.433328474821708 0.000969361674442 0.024910234567935
 2 -0.432221885533905 0.000001983967175 0.001106589287803
 3 -0.432219611394149 0.000000000008392 0.000002274139756
The value of root with absolute error 5e-09 is: -0.4322196113941486


### 1.2.3 Tìm nghiệm thoả mãn điều kiện sai số tương đối


Thiết lập điều kiện dừng theo công thức sai số: $\dfrac{|x_n - x^*|}{|x_n|} \leqslant \dfrac{M_2}{2 m_1} \cdot \dfrac{|x_n - x_{n-1}|^2}{|x_n|} < \eta$

với $m_1 = \min\limits_{[a,b]} |f'(x)|, \quad M_2 = \max\limits_{[a,b]} |f''(x)|$

Từ công thức sai số hậu nghiệm, ta có Điều kiện dừng: $\dfrac{|x_n - x_{n-1}|^2}{|x_n|} < 2 \eta \cdot \dfrac{m_1}{M_2}$

In [27]:
def newton_recursion_relative (f, df, d2f, a, b, eta):	
	M2 = max([np.abs(d2f(x)) for x in [a, b]]) #M2 is the maximum value of |f''(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print (f"m1 = {m1}, M2 = {M2}")

	#Starting values
	if f(a) * d2f(a) > 0:
		x = a
	elif f(b) * d2f(b) > 0:
		x = b
	
	results = []
	#Newton's method
	new_eta = 2 * eta * m1 / M2
	print(f"sigma_x = {new_eta}")

	i=0
	while True:
		# calculate the next value of x
		x_new = x - f(x) / df(x)
		current_sigma_x = (x-x_new)**2 / abs(x_new)

		results.append({
            'n': i,
            'x_(n+1)': x_new,
            'f(x_(n+1))': f(x_new),
            'delta_x=|x_n - x_(n-1)|': current_sigma_x
        })
		
		# update the value of interval 
		x = x_new
		if (f(x_new) == 0): 
			break

		#stop condition
		if current_sigma_x < new_eta:
			break
		else:
			i += 1

	#Print the final result
	df_results = pd.DataFrame(results)
	print(df_results.to_string(index=False))


	print(f"The value of root with relative error {eta} is: {x}")

Câu 11. Giải phương trình với nghiệm có sai lệch không vượt quá 0.05% bằng phương pháp tiếp tuyến.

a. $x^5-3x^3+2x^2-x+5=0$

In [28]:
f = lambda x: x**5 - 3*x**3 + 2*x**2 - x + 5
df = lambda x: 5*x**4 - 9*x**2 + 4*x - 1
d2f = lambda x: 20*x**3 - 18*x + 4

a = -3
b = -2

eta = 0.05 * pow(10, -2)

newton_recursion_relative (f, df, d2f, a, b, eta)

m1 = 35, M2 = 482
sigma_x = 7.261410788381743e-05
 n            x_(n+1)          f(x_(n+1))  delta_x=|x_n - x_(n-1)|
 0 -2.562700964630225 -39.343716790642659        0.074620663500990
 1 -2.291922162665554  -9.325586421256915        0.031991121159258
 2 -2.176107037027813  -1.236439447643527        0.006163825169559
 3 -2.155430322626288  -0.034437628281810        0.000198348568244
 4 -2.154820659887584  -0.000029256855267        0.000000172491689
The value of root with relative error 0.0005 is: -2.1548206598875836


b. $2^x-5x+\sin x = 0$

In [29]:
f = lambda x: 2**x - 5*x + np.sin(x)
df = lambda x: 2**x*np.log(2) - 5 + np.cos(x)
d2f = lambda x: 2**x*(np.log(2))**2 - np.sin(x)

a = 0
b = 1

eta = 0.05 * pow(10, -2)

newton_recursion_relative (f, df, d2f, a, b, eta)

m1 = 3.0734033330119694, M2 = 0.4804530139182014
sigma_x = 0.006396886363450362
 n           x_(n+1)        f(x_(n+1))  delta_x=|x_n - x_(n-1)|
 0 0.302402330736125 0.018998916612550        0.302402330736125
 1 0.308357003108829 0.000005205261387        0.000114990490596
The value of root with relative error 0.0005 is: 0.30835700310882896


c. $e^{-x}-x=0$

In [30]:
f = lambda x: np.e**(-x) - x
df = lambda x: -np.e**(-x) - 1
d2f = lambda x: np.e**(-x)

a = 0
b = 1

eta = 0.05 * pow(10, -2)

newton_recursion_relative (f, df, d2f, a, b, eta)

m1 = 1.3678794411714423, M2 = 1.0
sigma_x = 0.0013678794411714423
 n           x_(n+1)        f(x_(n+1))  delta_x=|x_n - x_(n-1)|
 0 0.500000000000000 0.106530659712633        0.500000000000000
 1 0.566311003197218 0.001304509806020        0.007764548313906
 2 0.567143165034862 0.000000196480472        0.000001221020311
The value of root with relative error 0.0005 is: 0.5671431650348623
